# Modelo de predicción de respaldo

Como el modelo principal no puede generalizar a calles no vistas (porque usa `segment_id` como identificador), en este notebook entrenamos un segundo modelo para cubrir el resto de segmentos de Chicago.

El modelo se entrena igual que el principal pero con una pequeña diferencia, sin `segment_id` y sin `speed_lag1`. Estas las sustituiremos por `highway_type`, que nos dirá el tipo de vía según OpenStreetMap.

Este modelo será menos preciso que el principal, ya que predice sin conocer nada específico del sitio ni su tráfico reciente. Es un modelo de respaldo, no se pretende igualar al principal.

## 0. Importación de paquetes

In [3]:
!pip install osmnx --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 5.3 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np
import pyproj
import osmnx as ox
import duckdb
import geopandas as gpd
from shapely.geometry import LineString
import lightgbm as lgb

from google.colab import drive
drive.mount('/content/drive')

# carpeta de Drive
data_clean = "/content/drive/MyDrive/TFM/data_clean"

Mounted at /content/drive


## 1. Emparejar segmentos con el grafo de OSM

Para saber que tipo de vía (`highway_type`) es cada segmento, primero de todo debemos encontrar su arista correspondiente (o aristas correspondientes) en el grafo de OpenStreetMap.

Para hacer este emparejamiento, construimos la línea de cada segmento, desde su punto de inicio a su punto de fin  (con un margen de grosor de 30 metros a cada lado para compensar el ruido GPS) y busamos todas las aristas de OSM de dentro de esa franja.


In [5]:
# cargamos grafo de OSM creado en grafo.ipynb
G = ox.load_graphml(filepath=f"{data_clean}/chicago_graph.graphml")

In [6]:
# emparejamos nuestros segmentos con su arista de osmnx

dim_segmentos = pd.read_parquet(f"{data_clean}/dim_segmentos.parquet")
G_proj = ox.project_graph(G)
# obtenemos sist. de coordenadas de osmnx
crs_proyectado = G_proj.graph["crs"]

dim_segmentos["geometry"] = dim_segmentos.apply(
    lambda r: LineString([
        (r["start_longitude"], r["start_latitude"]),
        (r["end_longitude"], r["end_latitude"])
    ]),
    axis=1
)

segs_gdf = gpd.GeoDataFrame(dim_segmentos, geometry="geometry", crs="EPSG:4326")
segs_gdf_proj = segs_gdf.to_crs(crs_proyectado)

# ponemos margen de grosor a la línea de cada segmento, para poder cruzarla con las aristas de OSM
margen_grosor = 30
segs_gdf_proj["geometry"] = segs_gdf_proj.buffer(margen_grosor)

print(segs_gdf_proj.shape)

(1046, 8)


In [7]:
edges_gdf_proj = ox.graph_to_gdfs(G_proj, nodes=False).reset_index()

emparejamiento = gpd.sjoin(
    edges_gdf_proj[["u", "v", "key", "geometry"]],
    segs_gdf_proj[["segment_id", "geometry"]],
    predicate="intersects",
    how="inner"
)

print(f"Aristas de OSM cubiertas por algún segmento: {emparejamiento[['u','v','key']].drop_duplicates().shape[0]}")
print(f"Total de emparejamientos (una arista puede tocar varios segmentos): {len(emparejamiento)}")

Aristas de OSM cubiertas por algún segmento: 16443
Total de emparejamientos (una arista puede tocar varios segmentos): 41331


El margen de grosor hace que una arista de OSM pueda estar dentro de la franja de más de uno de nuestros segmentos, esto nos puede dar problemas. Por lo tanto, para arreglarlo, para cada arista nos quedamos con el segmento cuya línea real (sin el margen de grosor) está más cerca.

In [8]:
segs_gdf_lineas = segs_gdf.to_crs(crs_proyectado)[["segment_id", "geometry"]].rename(columns={"geometry": "seg_geom"})
edges_geoms = edges_gdf_proj[["u", "v", "key", "geometry"]].rename(columns={"geometry": "edge_geom"})

emparejamiento = emparejamiento.merge(segs_gdf_lineas, on="segment_id")
emparejamiento = emparejamiento.merge(edges_geoms, on=["u", "v", "key"])

emparejamiento["dist_real"] = emparejamiento.apply(
    lambda r: r["seg_geom"].distance(r["edge_geom"]), axis=1
)

emparejamiento_final = (
    emparejamiento
    .sort_values("dist_real")
    .drop_duplicates(subset=["u", "v", "key"], keep="first")
)

print(f"Aristas totales cubiertas: {len(emparejamiento_final)}")
print(emparejamiento_final["dist_real"].describe())

Aristas totales cubiertas: 16443
count    16443.000000
mean         5.494864
std          6.642723
min          0.000000
25%          0.000000
50%          0.947862
75%         10.577355
max         29.966123
Name: dist_real, dtype: float64


In [9]:
# creamos y guardamos segmentos_monitorizados

segmentos_monitorizados = emparejamiento_final[["segment_id", "u", "v", "key"]].rename(
    columns={"u": "osm_u", "v": "osm_v", "key": "osm_key"}
)
segmentos_monitorizados.to_parquet(f"{data_clean}/segmentos_monitorizados.parquet")

Con esto cubrimos el recorrido real completo. 16.443 aristas en total, con una distancia mediana a la calle real de menos de 1 metro.

## 2. Tipo de vía de cada segmento

Ahora que ya sabemos qué aristas de OSM corresponden a cada segmento, calculamos el tipo de vía de una de ellas (como todas las aristas de un mismo segmento son la misma calle, deben ser del mismo tipo).

In [10]:
def get_tipo_via(segment_id, edges_del_segmento, G):
    for _, row in edges_del_segmento.iterrows():
        try:
            edge_data = G.edges[row["osm_u"], row["osm_v"], row["osm_key"]]
            hwy = edge_data.get("highway", "unknown")
            return hwy[0] if isinstance(hwy, list) else hwy
        except KeyError:
            continue
    return "unknown"

highway_por_segmento = []
for seg_id, grupo in segmentos_monitorizados.groupby("segment_id"):
    highway_por_segmento.append({
        "segment_id": seg_id,
        "highway_type": get_tipo_via(seg_id, grupo, G)
    })

segment_highway = pd.DataFrame(highway_por_segmento)
print(segment_highway["highway_type"].value_counts())

highway_type
residential       485
secondary         246
tertiary          144
primary            35
motorway           24
unclassified       20
motorway_link      16
secondary_link      6
emergency_bay       6
trunk_link          5
trunk               5
busway              3
primary_link        2
living_street       2
Name: count, dtype: int64


La mayoría de vías son residenciales (485), seguidas de las calles secundarias (246) y terciarias (144). En conjutno, podemos decir que es una red viaria predominantemente urbana.

In [11]:
# guardamos tipo de vía
segment_highway.to_parquet(f"{data_clean}/segment_highway_type.parquet")

## 3. Preparar variables y entrenar

Construimos el dataset del modelo uniendo `highway_type` desde la tabla de mapeo, sin seleccionar `segment_id` ni `speed_lag1`.

In [12]:
con = duckdb.connect()

In [13]:
fecha_corte = "2025-10-30 01:00:00"

In [14]:
train_df2 = con.sql(f"""
    SELECT
        CAST(f.avg_speed AS FLOAT) AS avg_speed,
        CAST(f.school_holiday AS TINYINT) AS school_holiday,
        CAST(f.festivo AS TINYINT) AS festivo,
        CAST(f.temperature_2m AS FLOAT) AS temperature_2m,
        CAST(f.rain AS FLOAT) AS rain,
        CAST(f.snowfall AS FLOAT) AS snowfall,
        CAST(f.relative_humidity_2m AS FLOAT) AS relative_humidity_2m,
        CAST(f.wind_speed_10m AS FLOAT) AS wind_speed_10m,
        CAST(f.partido AS TINYINT) AS partido,
        CAST(f.concierto AS TINYINT) AS concierto,
        f.obra_tipo,
        f.accidente_tipo,
        CAST(f.hora AS TINYINT) AS hora,
        CAST(f.dia_semana AS TINYINT) AS dia_semana,
        CAST(f.mes AS TINYINT) AS mes,
        CAST(f.es_finde AS TINYINT) AS es_finde,
        h.highway_type
    FROM '{data_clean}/dataset_final.parquet' f
    JOIN '{data_clean}/segment_highway_type.parquet' h
        ON f.segment_id = h.segment_id
    WHERE f.time_hour < '{fecha_corte}'
    USING SAMPLE 20 PERCENT (bernoulli)
""").df()

test_df2 = con.sql(f"""
    SELECT
        CAST(f.avg_speed AS FLOAT) AS avg_speed,
        CAST(f.school_holiday AS TINYINT) AS school_holiday,
        CAST(f.festivo AS TINYINT) AS festivo,
        CAST(f.temperature_2m AS FLOAT) AS temperature_2m,
        CAST(f.rain AS FLOAT) AS rain,
        CAST(f.snowfall AS FLOAT) AS snowfall,
        CAST(f.relative_humidity_2m AS FLOAT) AS relative_humidity_2m,
        CAST(f.wind_speed_10m AS FLOAT) AS wind_speed_10m,
        CAST(f.partido AS TINYINT) AS partido,
        CAST(f.concierto AS TINYINT) AS concierto,
        f.obra_tipo,
        f.accidente_tipo,
        CAST(f.hora AS TINYINT) AS hora,
        CAST(f.dia_semana AS TINYINT) AS dia_semana,
        CAST(f.mes AS TINYINT) AS mes,
        CAST(f.es_finde AS TINYINT) AS es_finde,
        h.highway_type
    FROM '{data_clean}/dataset_final.parquet' f
    JOIN '{data_clean}/segment_highway_type.parquet' h
        ON f.segment_id = h.segment_id
    WHERE f.time_hour >= '{fecha_corte}'
""").df()

print(train_df2.shape)
print(test_df2.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(9638264, 17)
(3693141, 17)


Obtenemos 9.634.004 filas para train y 3.693.141 filas para test.

In [15]:
# preparamos variables categoricas
columnas_categoricas2 = ["obra_tipo", "accidente_tipo", "hora", "dia_semana", "mes", "highway_type"]

for col in columnas_categoricas2:
    train_df2[col] = train_df2[col].astype("category")
    test_df2[col] = test_df2[col].astype("category")

Como `test_df` no ha sido muestreado pero `train_df` sí, es posible que algún valor de `obra_tipo`, `accidente_tipo` o `highway_type`que aparece en test no haya aparecido nunca en train. Vamos a comprobarlo por si acaso.

In [16]:
for col in ["obra_tipo", "accidente_tipo", "highway_type"]:
    en_test_no_train = set(test_df2[col].unique()) - set(train_df2[col].unique())
    print(f"{col}: categorías en test que no están en train -> {en_test_no_train}")

obra_tipo: categorías en test que no están en train -> set()
accidente_tipo: categorías en test que no están en train -> set()
highway_type: categorías en test que no están en train -> set()


Salen los tres conjuntos vacios, así que no hay problema.

In [17]:
# separamos X e y
X_train2 = train_df2.drop(columns=["avg_speed"])
y_train2 = train_df2["avg_speed"]

X_test2 = test_df2.drop(columns=["avg_speed"])
y_test2 = test_df2["avg_speed"]

print(X_train2.shape)
print(X_test2.shape)

(9638264, 16)
(3693141, 16)


In [18]:
# entrenamos modelo

train_data2 = lgb.Dataset(X_train2, label=y_train2, categorical_feature=columnas_categoricas2)
test_data2 = lgb.Dataset(X_test2, label=y_test2, categorical_feature=columnas_categoricas2, reference=train_data2)

params2 = {
    "objective": "regression",
    "metric": "mae",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "verbose": -1,
}

modelo2 = lgb.train(
    params2,
    train_data2,
    num_boost_round=500,
    valid_sets=[train_data2, test_data2],
    valid_names=["train", "test"],
    callbacks=[lgb.early_stopping(stopping_rounds=30), lgb.log_evaluation(50)],
)

Training until validation scores don't improve for 30 rounds
[50]	train's l1: 3.71937	test's l1: 3.58241
[100]	train's l1: 3.69197	test's l1: 3.55039
[150]	train's l1: 3.68599	test's l1: 3.54687
[200]	train's l1: 3.68351	test's l1: 3.54664
Early stopping, best iteration is:
[172]	train's l1: 3.6846	test's l1: 3.5465


In [19]:
# guardamos modelo
modelo2.save_model(f"{data_clean}/modelo2_trafico.txt")

Conseguimos un MAE final de 3.55km/h en test y 3.68km/h en train, casi el doble de error conseguido en el modelo principal. Tiene sentido ya que este modelo no tiene los datos que más información aportaban (`speed_lag1` y `segment_id`).

Aun así, los resultados son bastante buenos, un MAE de 3.54km/h sobre una media de ~25km/h es un 14% de error relativo, un porcentaje bastante razonable para un modelo que no conoce nada de la calle.



Por lo tanto, ahora tenemos dos modelos:
1. **Modelo principal:** con `segment_id` y `speed_lag1`, un MAE de 1.85km/h para los 1.046 segemntos monitorizados.
2. **Modelo de respaldo:** con `highway_type`, un MAE de 3.55km/h para cualquier calle de Chicago, monitorizada o no.